In [25]:
print("please enter the buying price of the car: vhigh, high, med, or low")   
buying = input("buying: ")   

print("please enter the safety of the car: high, med, or low")   
safety = input("safety: ")   

print("please enter the maint price of the car: vhigh, high, med, or low")   
maint = input("maint: ")  
#print(terms)- allows me to check the user input converted correctly 

please enter the buying price of the car: vhigh, high, med, or low
please enter the safety of the car: high, med, or low
please enter the maint price of the car: vhigh, high, med, or low


In [26]:
#Rules
row = {"buying": buying, "maint":maint, "safety":safety}
    #{"buying:" "low", "safety:" "high", "maint:" "med"}

facts = {f"{attribute}={value}" for attribute, value in row.items()}#each attribute, plus its value from row become a fact

rules = [#  list of tuples 

    ({("safety=low")}, ("class_value=unacc")),
    ({("maint=med"),("buying=low")}, ("class_value=good")),
    ({("maint=low"),("buying=low")}, ("class_value=good")),
    ({("safety=med"),("buying=high")}, ("class_value=acc")),
    ({("safety=med"),("buying=vhigh")}, ("class_value=acc")),
    ({("safety=med"),("buying=med")}, ("class_value=acc")),
    ({("safety=high"),("maint=med"),("buying=med")}, ("class_value=good")),
    ({("safety=high"),("maint=low")}, ("class_value=vgood")),
]

In [ ]:
#forward chaining

def forward_chain(facts, rules):#facts and rules are past into the forward chain function
    known = set(facts)
    changed = True
    while changed:#while changed is true the program will keep checking each rule.
        changed = False #while changed is true look for rules that satisfy, at the start of each iterate it resets when chnaged is = false
        for conditions, conclusion in rules:#iterates through every rule
            if conditions<=known and conclusion not in known:#checks to see if a conclusion is already in the known facts
                known.add(conclusion)# add conclusion to see if  both conditions are met
                changed = True
    return known
    #predict(facts, rules)

print(sorted(forward_chain(facts, rules)))


['buying=low', 'class_value=good', 'class_value=unacc', 'maint=low', 'safety=low']


In [28]:
#forward chaining with trace
def forward_chain_trace(facts, rules):
    known, trace = set(facts), []# stops the users facts from being altered
    changed = True
    while changed:
        changed = False
        for conditions, conclusion in rules:
            if conditions<= known and conclusion not in known:#checks conditions that are also present in known
                known.add(conclusion)
                trace.append((sorted(conditions), conclusion))#makes my conditions and conclusion into one because i am using tuples
                changed = True
    return known,trace

known, trace = forward_chain_trace(facts, rules)
for conditions, conclusion in trace:#only displays the rules that were a match
    print(f"{' + '.join(conditions)} -> {conclusion}") #displays conditions and the correct class value 


safety=low -> class_value=unacc
buying=low + maint=low -> class_value=good


In [29]:
#takes the attributes, passes them through the forward chain function and returns the derived class value of the car

def predict(row, rules, default = "class_value=unacc"):#only checks and takes on the first matching rule
    

    facts = {f"{attribute}={value}" for attribute, value in row.items()}#allows for the process to be seen
    derived = forward_chain(facts, rules)#stores results from forward chain into variable
   
    for facts in derived:#goes through each rule
        if facts.startswith("class_value="):
            return facts
    return default
print(predict(row, rules))

#displays process
known, trace = forward_chain_trace(facts, rules)
for conditions, conclusion in trace:#only displays the rules that were a match
    print(f"{' + '.join(conditions)} -> {conclusion}") #displays conditions and the correct class value 



        

class_value=unacc
safety=low -> class_value=unacc
buying=low + maint=low -> class_value=good


In [30]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
attributes = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class_value"]# names each attribute in the dataset using a list.
carData = pd.read_csv("car.data", names=attributes, header = None )#reads the attributes in the dataset matching them up to the attribute names in the list

train, test = train_test_split(#train,
    carData, test_size = 0.3, random_state=42 # 30% of the data set to test on and randomly shuffles data
)#30:70 test train split and shuffle

def classify(r):
    return predict(r,rules).split("=")[1]#removes class_value leaving the actual value to used for testing 

In [31]:
# calculates the percentage of the most common class value, if i were to make everything unacc, then i would have a 69% accurate algorithm
#calculates the baseline accuracy and the accuracy of my rules.
from collections import Counter
y_true = test["class_value"].tolist()
majority = Counter(y_true).most_common(1)[0][0]

baseline = sum(value == majority for value in y_true) /len(y_true)

y_pred = [classify(row) for n, row in test.iterrows()]

accuracy = accuracy_score(y_true, y_pred)

print(f"always predicts{majority}: {baseline:.27}")
print(f"my rules:{accuracy:.2f}")

always predictsunacc: 0.689788053949903612327432256
my rules:0.98


In [32]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_true = test["class_value"].tolist()
y_pred = [classify(row) for i, row in test.iterrows()]

print(f"Accuracy: {accuracy_score(y_true, y_pred):.2f}")
print(classification_report(y_true, y_pred, zero_division=0))
print(confusion_matrix(y_true, y_pred, labels=sorted(set(y_true))))

Accuracy: 0.98
              precision    recall  f1-score   support

         acc       0.99      1.00      1.00       118
        good       1.00      0.63      0.77        19
       unacc       1.00      0.99      0.99       358
       vgood       0.71      1.00      0.83        24

    accuracy                           0.98       519
   macro avg       0.92      0.91      0.90       519
weighted avg       0.98      0.98      0.98       519

[[118   0   0   0]
 [  1  12   0   6]
 [  0   0 354   4]
 [  0   0   0  24]]
